<a href="https://colab.research.google.com/github/janaina-passos/codehealth-ci-cd-janaina-diogopassos/blob/main/Oficina_Modelos_de_Linguagem_(WTT_2026)_(SALVE_COMO_C%C3%93PIA).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A "Mágica" por trás do ChatGPT
## Criando um Modelo de Linguagem do Zero

**Prof. Me. Lucas C. Figueiredo** · Datalab · FCI Mackenzie

---

> Vamos alternar entre **slides** (conceito) e **código** (live-coding).
> Acompanhe no notebook e rode cada célula junto comigo.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import json
import random
import pandas as pd
import unicodedata

random.seed(42)
np.random.seed(42)

print("Setup pronto. NumPy:", np.__version__)

## 1. Nosso Corpus: Nomes Brasileiros

- Fonte: **Nomes no Brasil - IBGE Censo 2022** (GitHub)
- 10.000 nomes masculinos + 10.000 femininos
- Cada entrada tem nome e frequência no censo
- Usaremos os nomes como **sequências de caracteres**

Por que nomes?
- Curtos: modelo simples já captura a estrutura
- Têm padrões reconhecíveis
- Resultados fáceis de avaliar sem métricas complicadas

In [ ]:
# Fallback mínimo caso o download falhe
NOMES_FALLBACK = [
    ("maria", 1000), ("jose", 900), ("ana", 700), ("joao", 650), ("antonio", 600),
    ("francisco", 550), ("carlos", 500), ("paulo", 480), ("pedro", 460), ("lucas", 430)
]

def limpar(s):
    """Remove acentos e normaliza para minúsculas."""
    s = unicodedata.normalize("NFD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = "".join(c for c in s if c.isalpha())
    return s.lower()

def montar_corpus_com_frequencia(df_fem, df_mas, max_por_genero=5000):
    """Monta corpus de nomes + frequências a partir de CSVs do GitHub/IBGE."""
    fem = df_fem.copy()
    mas = df_mas.copy()

    nome_col_f = next((c for c in fem.columns if c.lower() == "nome"), fem.columns[0])
    nome_col_m = next((c for c in mas.columns if c.lower() == "nome"), mas.columns[0])
    freq_col_f = next((c for c in fem.columns if "freq" in c.lower()), None)
    freq_col_m = next((c for c in mas.columns if "freq" in c.lower()), None)

    if freq_col_f is None or freq_col_m is None:
        raise ValueError("Nao encontrei coluna de frequencia nos CSVs.")

    fem = fem[[nome_col_f, freq_col_f]].rename(columns={nome_col_f: "nome", freq_col_f: "frequencia"})
    mas = mas[[nome_col_m, freq_col_m]].rename(columns={nome_col_m: "nome", freq_col_m: "frequencia"})

    #fem = fem.head(max_por_genero)
    #mas = mas.head(max_por_genero)

    df = pd.concat([fem, mas], ignore_index=True)
    df["nome"] = df["nome"].map(limpar)
    df["frequencia"] = pd.to_numeric(df["frequencia"], errors="coerce").fillna(0)

    df = df[df["nome"].str.len() >= 2]
    df = df.groupby("nome", as_index=False)["frequencia"].sum()
    df["peso"] = (df["frequencia"] / df["frequencia"].max() * 1000).clip(lower=1)

    return df.sort_values("frequencia", ascending=False).reset_index(drop=True)

# Carregar CSVs do GitHub
url_fem = "https://raw.githubusercontent.com/MedidaSP/nomes-brasileiros-ibge/master/ibge-fem-10000.csv"
url_mas = "https://raw.githubusercontent.com/MedidaSP/nomes-brasileiros-ibge/master/ibge-mas-10000.csv"

df_fem = pd.read_csv(url_fem)
df_mas = pd.read_csv(url_mas)
print(f"Femininos: {len(df_fem)}   Masculinos: {len(df_mas)}")

try:
    df_nomes = montar_corpus_com_frequencia(df_fem, df_mas, max_por_genero=5000)
    print(f"Dataset carregado com {len(df_nomes)} nomes únicos.")
except Exception as e:
    print(f"Falha ({e}). Usando fallback.")
    df_nomes = pd.DataFrame(NOMES_FALLBACK, columns=["nome", "frequencia"])
    df_nomes["peso"] = df_nomes["frequencia"]

nomes = df_nomes["nome"].tolist()
nome_para_peso = dict(zip(df_nomes["nome"], df_nomes["peso"]))

print(f"Exemplos: {nomes[:10]}")

In [ ]:
# Visualizar amostra do dataset
df_fem.head()

## 2. Vocabulário: caractere → número

- O computador não consegue processar texto diretamente
- Traduzimos cada caractere para um número (índice)
- `"maria"` vira a sequência `[0, 13, 1, 18, 9, 1, 0]`
- Adicionamos um token especial `.` para marcar **início e fim**
- O que muda do nosso modelo pro GPT é o **tamanho do vocabulário**

In [ ]:
# Vocabulário: todas as letras únicas + '.' (início/fim)
chars = sorted(set("".join(nomes)))
chars = ["."] + chars  # '.' é o token 0

stoi = {c: i for i, c in enumerate(chars)}  # Dicionário de string to int
itos = {i: c for i, c in enumerate(chars)}  # Dicionário de int to string

V = len(chars)
print(f"Vocabulário ({V} tokens): {chars}")
print(f"\nExemplo de mapeamento:")
print(f"  'maria' -> {[stoi[c] for c in 'maria']}")
print(f"  'a' -> {stoi['a']}, '.' -> {stoi['.']}")

## 3. Bigram: contando pares de caracteres

A pergunta central: **dado o caractere anterior, qual a probabilidade de cada token ser o próximo?**

$$P(\text{próximo} \mid \text{anterior})$$

É como o autocompletar do celular, mas que só lembra da **última letra**.

**Construindo o modelo:** para cada par consecutivo no corpus, somamos 1 na tabela.

Exemplo com `.maria.`:  `.→m`, `m→a`, `a→r`, `r→i`, `i→a`, `a→.`

In [ ]:
# Tabela de contagem (ponderada com log da frequência)
N = np.zeros((V, V), dtype=np.float64)

for nome in nomes:
    peso = np.log1p(float(nome_para_peso.get(nome, 1.0)))
    chars_nome = ["."] + list(nome) + ["."]

    # A iteração com zip funciona da seguinte forma:
    # -- Para o nome "maria", chars_nome seria ['.', 'm', 'a', 'r', 'i', 'a', '.']
    # -- O zip(chars_nome, chars_nome[1:]) cria pares consecutivos:
    # - ('.', 'm')
    # - ('m', 'a')
    # - ('a', 'r')
    # - ('r', 'i')
    # - ('i', 'a')
    # -- E assim por diante para cada nome, contando os pares de caracteres e acumulando o peso (log da frequencia) correspondente.

    for ch1, ch2 in zip(chars_nome, chars_nome[1:]):
        i1, i2 = stoi[ch1], stoi[ch2]
        N[i1, i2] += peso

print(f"Matriz de contagem: {N.shape}")
print(f"Total de pares contados: {N.sum():.1f}")

In [ ]:
# Heatmap da matriz de co-ocorrência
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(N, cmap="Reds")
ax.set_xticks(range(V))
ax.set_yticks(range(V))
ax.set_xticklabels(chars, fontsize=8)
ax.set_yticklabels(chars, fontsize=8)
ax.set_xlabel("caractere seguinte")
ax.set_ylabel("caractere anterior")
ax.set_title("Matriz de Co-ocorrência (contagem)")
fig.colorbar(im, label="Contagem")
plt.tight_layout()
plt.show()

## 4. De contagens para probabilidades

Contagens não são probabilidades. Para virar probabilidade, **dividimos cada linha pelo total da linha**.

- Agora cada linha é uma **distribuição de probabilidade** (soma = 1)
- "Depois de `a`, em X% das vezes vem `n`"
- Temos um modelo que aprendeu a probabilidade de cada par sequencial

In [ ]:
# Normalização: cada linha vira uma distribuição de probabilidade
P = (N + 1).astype(np.float32)  # +1 = Suavização de Laplace
P = P / P.sum(axis=1, keepdims=True)

print(f"Shape: {P.shape}")
print(f"Soma da linha 'a': {P[stoi['a']].sum():.4f}")

print("\nTop 5 transições depois de 'a':")
top = np.argsort(P[stoi['a']])[::-1][:5]
for idx in top:
    print(f"  a → {itos[idx]}: {P[stoi['a'], idx]:.3f}")

# Heatmap de probabilidades
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(P, cmap="Reds")
ax.set_xticks(range(V))
ax.set_yticks(range(V))
ax.set_xticklabels(chars, fontsize=8)
ax.set_yticklabels(chars, fontsize=8)
ax.set_xlabel("caractere seguinte")
ax.set_ylabel("caractere anterior")
ax.set_title("Probabilidades do Bigram")
fig.colorbar(im, label="Probabilidade")
plt.tight_layout()
plt.show()

## 5. Gerando nomes novos

Com a tabela de probabilidades, geramos um nome **passo a passo**:
1. Começar com `.` (início)
2. Sortear o **próximo caractere** pela distribuição
3. Alimentar de volta e repetir
4. Parar quando cair em `.` (fim)

Esse loop é o **mesmo** que o ChatGPT faz. Token por token.

In [ ]:
def gerar_nome(P, stoi, itos, max_len=13):
    """Gera um nome amostrando do bigram."""
    out = []
    idx = stoi["."]
    for _ in range(max_len):
        probs = P[idx] # Pega a distribuição de probabilidade para caracteres após .
        idx = np.random.choice(len(probs), p=probs) # Sorteia um dos caracteres com base na prob.
        if idx == stoi["."]: # Se for sorteado . o nome acaba
            break
        out.append(itos[idx])
    return "".join(out)

# Gera 20 nomes
print("Nomes gerados pelo bigram:")
for i in range(20):
    print(f"  {i+1:2d}. {gerar_nome(P, stoi, itos)}")

## 6. Avaliando o modelo: Loss (NLL)

Podemos avaliar no "olhômetro", mas precisamos de um **número objetivo**.

- Modelo bom → atribui probabilidade **alta** a nomes reais → loss **baixa**
- Modelo ruim → probabilidade baixa → loss **alta**

Esse número é a **Negative Log-Likelihood (NLL)**: a métrica padrão para modelos de linguagem.

In [ ]:
def loss_bigram(P, nomes, stoi):
    """Calcula NLL média sobre o corpus."""
    log_likelihood = 0.0
    n_pares = 0
    for nome in nomes:
        chars_nome = ["."] + list(nome) + ["."]
        for ch1, ch2 in zip(chars_nome, chars_nome[1:]):
            i1, i2 = stoi[ch1], stoi[ch2]
            log_likelihood += np.log(P[i1, i2])
            n_pares += 1
    return -log_likelihood / n_pares


nll_bigram = loss_bigram(P, nomes, stoi)
print(f"Loss do bigram:            {nll_bigram:.4f}")

# Baseline: modelo uniforme (chuta tudo igual)
P_uniforme = np.ones((V, V)) / V

# Geração com modelo uniforme vs bigram — comparação visual
print("\n" + "=" * 50)
print(f"{'Uniforme (Chute aleatório)':<30} {'Bigram':<30}")
print("=" * 50)
for i in range(15):
    nome_uni = gerar_nome(P_uniforme, stoi, itos)
    nome_bi  = gerar_nome(P, stoi, itos)
    print(f"{nome_uni:<30} {nome_bi:<30}")

nll_uniforme = loss_bigram(P_uniforme, nomes, stoi)
print(f"\nLoss do chute aleatório (baseline): {nll_uniforme:.4f}")
print(f"\nO bigram é {nll_uniforme/nll_bigram:.2f}x melhor que chutar aleatório.")



## 7. O modelo é espelho do corpus

E se o corpus fosse apenas de nomes masculinos? Ou nomes em inglês?

- Corpus só com nomes masculinos → modelo só gera nomes masculinos
- Corpus em inglês → modelo só gera nomes em inglês
- Corpus com vieses → modelo incorpora os vieses

Vamos treinar **4 bigrams separados** e comparar.

In [ ]:
# Preparar 4 corpus separados

# 1) Feminino BR
nome_col_f = next((c for c in df_fem.columns if c.lower() == "nome"), df_fem.columns[0])
nomes_fem = [limpar(n) for n in df_fem[nome_col_f].tolist()]
nomes_fem = sorted(set(n for n in nomes_fem if len(n) >= 2))

# 2) Masculino BR
nome_col_m = next((c for c in df_mas.columns if c.lower() == "nome"), df_mas.columns[0])
nomes_mas = [limpar(n) for n in df_mas[nome_col_m].tolist()]
nomes_mas = sorted(set(n for n in nomes_mas if len(n) >= 2))

# 3) Brasileiro Geral = união feminino + masculino
nomes_br = sorted(set(nomes_fem) | set(nomes_mas))

# 4) Inglês (internacional, bem variado)
URL_MAKEMORE_NAMES = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
try:
    df_en = pd.read_csv(URL_MAKEMORE_NAMES, header=None, names=["nome"])
    nomes_ingles = sorted(set(
        limpar(n) for n in df_en["nome"].tolist() if len(limpar(n)) >= 2
    ))
except Exception as e:
    print(f"Falha ao carregar names.txt ({e}). Usando fallback.")
    nomes_ingles = ["emma", "olivia", "ava", "liam", "noah", "oliver", "james"]

print(f"Feminino BR:       {len(nomes_fem)} nomes  — ex: {nomes_fem[:5]}")
print(f"Masculino BR:      {len(nomes_mas)} nomes  — ex: {nomes_mas[:5]}")
print(f"Brasileiro Geral:  {len(nomes_br)} nomes  — ex: {nomes_br[:5]}")
print(f"Inglês:            {len(nomes_ingles)} nomes  — ex: {nomes_ingles[:5]}")

In [ ]:
# Treinar um bigram para cada corpus e comparar

def treinar_bigram(lista_nomes):
    """Treina bigram e retorna (P, stoi, itos, V)."""
    chs = sorted(set("".join(lista_nomes)))
    chs = ["."] + chs
    st = {c: i for i, c in enumerate(chs)}
    it = {i: c for i, c in enumerate(chs)}
    v = len(chs)
    N = np.zeros((v, v), dtype=np.int32)
    for nome in lista_nomes:
        cs = ["."] + list(nome) + ["."]
        for c1, c2 in zip(cs, cs[1:]):
            N[st[c1], st[c2]] += 1
    P = (N + 1).astype(np.float32)
    P = P / P.sum(axis=1, keepdims=True)
    return P, st, it, v

def gerar_nome_local(P, stoi, itos, max_len=13):
    out = []
    idx = stoi["."]
    for _ in range(max_len):
        probs = P[idx]
        idx = np.random.choice(len(probs), p=probs)
        if idx == stoi["."]:
            break
        out.append(itos[idx])
    return "".join(out)

# Treinar os 4 bigrams
P_fem, stoi_fem, itos_fem, V_fem = treinar_bigram(nomes_fem)
P_mas, stoi_mas, itos_mas, V_mas = treinar_bigram(nomes_mas)
P_br, stoi_br, itos_br, V_br = treinar_bigram(nomes_br)
P_en, stoi_en, itos_en, V_en = treinar_bigram(nomes_ingles)

# Gerar e comparar lado a lado
N_GERAR = 15
gen_fem = [gerar_nome_local(P_fem, stoi_fem, itos_fem) for _ in range(N_GERAR)]
gen_mas = [gerar_nome_local(P_mas, stoi_mas, itos_mas) for _ in range(N_GERAR)]
gen_br  = [gerar_nome_local(P_br, stoi_br, itos_br) for _ in range(N_GERAR)]
gen_en  = [gerar_nome_local(P_en, stoi_en, itos_en) for _ in range(N_GERAR)]

print(f"{'#':>3}  {'Feminino BR':<16} {'Masculino BR':<16} {'Brasil Geral':<16} {'Inglês':<16}")
print("-" * 78)
for i in range(N_GERAR):
    print(f"{i+1:3d}  {gen_fem[i]:<16} {gen_mas[i]:<16} {gen_br[i]:<16} {gen_en[i]:<16}")

**Mesmo código. Mesmo algoritmo. Corpus diferente → resultado diferente.**

Note como os nomes femininos tendem a terminar em **a**, os masculinos em consoantes ou **o**, o corpus brasileiro geral mistura padrões dos dois, e os ingleses têm padrões completamente distintos.

O modelo não "sabe" de gênero, ele só aprendeu a **estatística do corpus** que recebeu.

Imagina isso com **toda a internet em vez de uma lista de nomes**. É o que LLMs atuais fazem.

## 8. Aumentando o contexto: Trigram

O bigram só lembra do **último caractere**. Quando vai gerar a 5ª letra, esqueceu as 4 primeiras.

E se o modelo olhasse **mais caracteres para trás**?

Cada token a mais de contexto **multiplica** o tamanho da tabela por V (27)...

In [ ]:
def treinar_trigram(lista_nomes):
    """Treina trigram e retorna (P3, stoi, itos, V, N3)."""
    chs = sorted(set("".join(lista_nomes)))
    chs = ["."] + chs
    st = {c: i for i, c in enumerate(chs)}
    it = {i: c for i, c in enumerate(chs)}
    v = len(chs)
    N3 = np.zeros((v, v, v), dtype=np.int32)
    for nome in lista_nomes:
        cs = [".", "."] + list(nome) + ["."]
        for c1, c2, c3 in zip(cs, cs[1:], cs[2:]):
            N3[st[c1], st[c2], st[c3]] += 1
    P3 = (N3 + 1).astype(np.float32)
    P3 = P3 / P3.sum(axis=2, keepdims=True)
    return P3, st, it, v, N3

def gerar_nome_trigram(P3, stoi, itos, max_len=20):
    out = []
    i1, i2 = stoi["."], stoi["."]
    for _ in range(max_len):
        probs = P3[i1, i2]
        i3 = np.random.choice(len(probs), p=probs)
        if i3 == stoi["."]:
            break
        out.append(itos[i3])
        i1, i2 = i2, i3
    return "".join(out)

# Treinar trigram para cada corpus
corpus_tri = {
    "Feminino BR":      nomes_fem,
    "Masculino BR":     nomes_mas,
    "Brasileiro Geral": nomes_br,
    "Inglês":           nomes_ingles,
}
trigrams = {}
for label, lista in corpus_tri.items():
    P3, st, it, v, N3 = treinar_trigram(lista)
    trigrams[label] = (P3, st, it, v, N3)
    esp = 100 * (N3 == 0).sum() / N3.size
    print(f"{label:16s}  V={v:2d}  shape={P3.shape}  esparsidade={esp:.1f}%")

In [ ]:
# Geração lado a lado com trigram
#np.random.seed(42)
n_amostras = 30

linhas = []
for i in range(n_amostras):
    row = []
    for label in corpus_tri:
        P3, st, it, v, _ = trigrams[label]
        row.append(gerar_nome_trigram(P3, st, it))
    linhas.append(row)

header = list(corpus_tri.keys())
col_w = [max(len(h), max(len(r[j]) for r in linhas)) + 2 for j, h in enumerate(header)]
sep = "+" + "+".join("-" * w for w in col_w) + "+"
head = "|" + "|".join(f" {h:<{w-1}}" for h, w in zip(header, col_w)) + "|"
print(sep)
print(head)
print(sep)
for row in linhas:
    print("|" + "|".join(f" {c:<{w-1}}" for c, w in zip(row, col_w)) + "|")
print(sep)

In [ ]:
# O problema: a tabela explode exponencialmente
V_max = max(v for _, _, _, v, _ in trigrams.values())
print(f"Maior vocabulário: V = {V_max}\n")
print(f"{'Modelo':<10} {'Tamanho da tabela':>20}")
print("-" * 32)
for n in range(1, 6):
    tamanho = V_max ** (n + 1)
    print(f"{n}-gram     {tamanho:>20,}")

total_chars = sum(sum(len(n) for n in lista) for lista in corpus_tri.values())
print(f"\nNossos corpus têm ~{total_chars:,} caracteres no total.")
print("Não dá pra preencher uma tabela dessas com tão poucos dados.")

In [ ]:
# Comparação de loss: bigram vs trigram
def loss_trigram(P3, nomes, stoi):
    ll, n = 0.0, 0
    for nome in nomes:
        cs = [".", "."] + list(nome) + ["."]
        for ch1, ch2, ch3 in zip(cs, cs[1:], cs[2:]):
            ll += np.log(P3[stoi[ch1], stoi[ch2], stoi[ch3]])
            n += 1
    return -ll / n

def loss_bigram_generic(P, nomes, stoi):
    ll, n = 0.0, 0
    for nome in nomes:
        cs = ["."] + list(nome) + ["."]
        for ch1, ch2 in zip(cs, cs[1:]):
            ll += np.log(P[stoi[ch1], stoi[ch2]])
            n += 1
    return -ll / n

print(f"{'Corpus':<16} {'Bigram':>8} {'Trigram':>8} {'Melhoria':>9}")
print("-" * 46)

bigrams_dict = {
    "Feminino BR":      (P_fem, stoi_fem),
    "Masculino BR":     (P_mas, stoi_mas),
    "Brasileiro Geral": (P_br, stoi_br),
    "Inglês":           (P_en, stoi_en),
}
for label in corpus_tri:
    P_bi, st_bi = bigrams_dict[label]
    P3, st_tri, _, _, _ = trigrams[label]
    lista = corpus_tri[label]
    nll_bi = loss_bigram_generic(P_bi, lista, st_bi)
    nll_tri = loss_trigram(P3, lista, st_tri)
    print(f"{label:<16} {nll_bi:>8.4f} {nll_tri:>8.4f} {nll_bi - nll_tri:>+9.4f}")

## 9. A solução: Rede Neural (MLP)

O problema não é o tamanho da tabela: é que ela fica **vazia** (esparsidade).

**A saída:** em vez de guardar uma tabela, vamos **aprender uma função**.

Essa função aprendida é uma **rede neural**. Cada peso ajustado é o que se chama de **parâmetro**.

> ⚠️ **Esta seção é demonstração.** O ponto é mostrar que funciona com NumPy puro, o resultado é melhor, e o modelo tem parâmetros ajustáveis, exatamente como um GPT, só que minúsculo. Não se preocupe se não entender o código.

In [ ]:
# Construção do dataset: janela de 3 caracteres -> próximo caractere
JANELA = 3

def construir_dataset(nomes, stoi, janela=3):
    X, Y = [], []
    for nome in nomes:
        cs = ["."] * janela + list(nome) + ["."]
        for i in range(len(cs) - janela):
            contexto = [stoi[c] for c in cs[i:i+janela]]
            alvo = stoi[cs[i+janela]]
            X.append(contexto)
            Y.append(alvo)
    return np.array(X), np.array(Y)

# Datasets para cada corpus
corpus_mlp = {
    "Feminino BR":      (nomes_fem, stoi_fem, itos_fem, V_fem),
    "Masculino BR":     (nomes_mas, stoi_mas, itos_mas, V_mas),
    "Brasileiro Geral": (nomes_br, stoi_br, itos_br, V_br),
    "Inglês":           (nomes_ingles, stoi_en, itos_en, V_en),
}
datasets = {}
for label, (lista, st, it, v) in corpus_mlp.items():
    X, Y = construir_dataset(lista, st, JANELA)
    datasets[label] = (X, Y)
    print(f"{label:16s}  X={str(X.shape):>14s}  Y={str(Y.shape):>10s}  V={v}")

In [ ]:
# MLP simples: embedding + camada oculta + softmax
DIM_EMB = 8
DIM_HIDDEN = 64

def init_mlp(V, janela=JANELA, dim_emb=DIM_EMB, dim_hid=DIM_HIDDEN):
    C  = np.random.randn(V, dim_emb) * 0.1
    W1 = np.random.randn(janela * dim_emb, dim_hid) * 0.1
    b1 = np.zeros(dim_hid)
    W2 = np.random.randn(dim_hid, V) * 0.1
    b2 = np.zeros(V)
    return C, W1, b1, W2, b2

V_ex = max(v for _, _, _, v in corpus_mlp.values())
params_ex = sum(p.size for p in init_mlp(V_ex))
print(f"Parâmetros do MLP (V={V_ex}): {params_ex:,}")
print(f"\nPra comparar:")
print(f"  Bigram (tabela): {V_ex*V_ex:,} parâmetros")
print(f"  GPT-2 small:     124,000,000 parâmetros")
print(f"  GPT-4 (estimado): 1,800,000,000,000 parâmetros")

In [ ]:
# Forward + backward manuais (sem PyTorch!)
def forward(X_batch, Y_batch, C, W1, b1, W2, b2):
    emb = C[X_batch]                              # (B, JANELA, DIM_EMB)
    h_in = emb.reshape(emb.shape[0], -1)          # (B, JANELA*DIM_EMB)
    h_pre = h_in @ W1 + b1                        # (B, DIM_HIDDEN)
    h = np.tanh(h_pre)                            # (B, DIM_HIDDEN)
    logits = h @ W2 + b2                          # (B, V)
    logits = logits - logits.max(axis=1, keepdims=True)
    counts = np.exp(logits)
    probs = counts / counts.sum(axis=1, keepdims=True)
    n = X_batch.shape[0]
    loss = -np.log(probs[np.arange(n), Y_batch] + 1e-10).mean()
    cache = (X_batch, Y_batch, emb, h_in, h_pre, h, probs)
    return loss, cache

def backward(cache, C, W1, b1, W2, b2):
    X_batch, Y_batch, emb, h_in, h_pre, h, probs = cache
    n = X_batch.shape[0]
    dlogits = probs.copy()
    dlogits[np.arange(n), Y_batch] -= 1
    dlogits /= n
    dW2 = h.T @ dlogits
    db2 = dlogits.sum(axis=0)
    dh = dlogits @ W2.T
    dh_pre = dh * (1 - h**2)
    dW1 = h_in.T @ dh_pre
    db1 = dh_pre.sum(axis=0)
    dh_in = dh_pre @ W1.T
    demb = dh_in.reshape(emb.shape)
    dC = np.zeros_like(C)
    for i in range(n):
        for j in range(JANELA):
            dC[X_batch[i, j]] += demb[i, j]
    return dC, dW1, db1, dW2, db2

print("Forward e backward implementados. Pronto pra treinar.")

In [ ]:
# Treino: gradiente descendente em mini-batches — 4 corpus
LR = 0.1
BATCH = 64
EPOCHS = 15

modelos = {}

for label, (lista, st, it, v) in corpus_mlp.items():
    print(f"\n{'='*52}")
    print(f"Treinando: {label}  (V={v})")
    print(f"{'='*52}")
    np.random.seed(42)
    C, W1, b1, W2, b2 = init_mlp(v)
    X, Y = datasets[label]

    losses = []
    for epoch in range(EPOCHS):
        idx = np.random.permutation(len(X))
        X_s, Y_s = X[idx], Y[idx]
        epoch_loss, n_batches = 0, 0
        for i in range(0, len(X), BATCH):
            xb, yb = X_s[i:i+BATCH], Y_s[i:i+BATCH]
            loss, cache = forward(xb, yb, C, W1, b1, W2, b2)
            dC, dW1, db1, dW2, db2 = backward(cache, C, W1, b1, W2, b2)
            C  -= LR * dC
            W1 -= LR * dW1
            b1 -= LR * db1
            W2 -= LR * dW2
            b2 -= LR * db2
            epoch_loss += loss
            n_batches += 1
        avg = epoch_loss / n_batches
        losses.append(avg)
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:2d}/{EPOCHS}  loss = {avg:.4f}")

    modelos[label] = (C, W1, b1, W2, b2, losses)

# Curvas de treino
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharey=True)
axes = axes.flatten()
colors = ['#c81e2d', '#1e5fc8', '#0b7a5c', '#e08a00']
for ax, ((label, _), color) in zip(axes, zip(corpus_mlp.items(), colors)):
    _, _, _, _, _, ls = modelos[label]
    ax.plot(ls, color=color)
    ax.set_title(label)
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Loss (NLL)")
axes[2].set_ylabel("Loss (NLL)")
fig.suptitle("Treino do MLP — 4 corpus", y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# Geração com MLP — comparação 4 corpus
def gerar_mlp(C, W1, b1, W2, b2, stoi, itos, V, janela=JANELA, max_len=20):
    out = []
    contexto = [stoi["."]] * janela
    for _ in range(max_len):
        x = np.array([contexto])
        emb = C[x].reshape(1, -1)
        h = np.tanh(emb @ W1 + b1)
        logits = h @ W2 + b2
        logits = logits - logits.max()
        probs = np.exp(logits) / np.exp(logits).sum()
        idx = np.random.choice(V, p=probs[0])
        if idx == stoi["."]:
            break
        out.append(itos[idx])
        contexto = contexto[1:] + [idx]
    return "".join(out)

np.random.seed(42)
n_amostras = 15
linhas = []
for i in range(n_amostras):
    row = []
    for label, (lista, st, it, v) in corpus_mlp.items():
        C, W1, b1, W2, b2, _ = modelos[label]
        row.append(gerar_mlp(C, W1, b1, W2, b2, st, it, v))
    linhas.append(row)

header = list(corpus_mlp.keys())
col_w = [max(len(h), max(len(r[j]) for r in linhas)) + 2 for j, h in enumerate(header)]
sep = "+" + "+".join("-" * w for w in col_w) + "+"
head = "|" + "|".join(f" {h:<{w-1}}" for h, w in zip(header, col_w)) + "|"
print("Nomes gerados pelo MLP:")
print(sep)
print(head)
print(sep)
for row in linhas:
    print("|" + "|".join(f" {c:<{w-1}}" for c, w in zip(row, col_w)) + "|")
print(sep)

**Compare os nomes MLP com os do bigram e trigram acima.** O MLP gera nomes mais coerentes, porque consegue olhar 3 caracteres de contexto e **generaliza** para combinações que não viu nos dados.

## 10. Do bigram ao GPT

O que construímos hoje:
* 1. **Bigram** — tabela de contagens, olha 1 caractere
* 2. **Trigram** — tabela maior, olha 2 caracteres, mas explode
* 3. **MLP** — função aprendida, olha N caracteres com janela fixa

O caminho continua:
* 4. **RNN/LSTM** — olha sequência inteira (com dificuldade em contexto longo)
* 5. **Attention** — olha tudo ao mesmo tempo, eficiente
* 6. **Transformer** — empilhamento de attention (a arquitetura do GPT)
* 7. **Treino em escala** — bilhões de parâmetros, terabytes de texto

**O princípio nunca muda: prever o próximo token a partir do que veio antes.**

O que muda é **como** o modelo representa contexto, o tamanho do modelo, e os dados de treino.

---

### Para continuar
- **Slides da Oficina**: [Slides no Canva](https://canva.link/3wj2lr57zcvicv8)
- **LLMs explained briefly**: [YouTube (3blue1brown)](https://www.youtube.com/watch?v=LPZh9BOjkQs)
- **makemore** (Andrej Karpathy): [Github](https://github.com/karpathy/makemore) e [Playlist no Youtube](https://youtu.be/PaCmpygFfXo)
- **Datalab FCI Mackenzie**: [datalab.mackenzie.br](https://datalab.mackenzie.br)
- **MackAI (Liga de IA)**: [instagram.com/mackenzie.iacd](https://instagram.com/mackenzie.iacd)

**Contato:** lucas.figueiredo@mackenzie.br